# Clasificación de géneros musicales con MFCC, RNN y LSTM

Este notebook integra las partes complementarias de **TP_3** y **Ap1_tp3** y las adapta a la consigna: cada canción se representa como una secuencia de **40 MFCC por frame**, y se comparan una RNN clásica y una LSTM entrenadas desde cero.

Del TP_3 se conservan la partición estratificada 80/10/10, la extracción paralela con caché y la separación entre validación y test. De Ap1 se conserva la comparación RNN/LSTM, la atención temporal y el análisis de métricas por género. La rama CNN/CRNN no se incluye en esta versión porque la entrada requerida para los modelos recurrentes es directamente `(Batch, T, 40)`.

**Flujo:** WAV → mono a 22 050 Hz → MFCC → transposición a `(T, 40)` → normalización por canción y coeficiente → padding/truncation a 1300 frames → RNN/LSTM.

## 1. Entorno y reproducibilidad

Está pensado para Google Colab. La celda siguiente descarga el GTZAN usando el mismo archivo de Google Drive que se utilizaba en TP_3. Si el dataset ya está en otra ubicación, se puede modificar `DATASET_PATH` después de la descarga.

In [ ]:
import os
import random
import hashlib
import zipfile
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {DEVICE}")
print(f"PyTorch: {torch.__version__} | Librosa: {librosa.__version__}")

In [ ]:
# Si se ejecuta en Colab y no existe el dataset, descargarlo.
DATASET_ARCHIVE = Path("dataset_music.zip")
DATASET_DIR = Path("datos_music")
DATASET_PATH = DATASET_DIR / "genres_original"

if not DATASET_PATH.is_dir():
    if not DATASET_ARCHIVE.exists():
        import gdown
        url = "https://drive.google.com/uc?id=1txyVxGVezQ4fcHqFi-n3gcWKEYQexu1_"
        gdown.download(url, str(DATASET_ARCHIVE), quiet=False)

    DATASET_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(DATASET_ARCHIVE, "r") as archive:
        archive.extractall(DATASET_DIR)

if not DATASET_PATH.is_dir():
    raise FileNotFoundError(
        f"No se encontró la carpeta GTZAN: {DATASET_PATH}. "
        "Actualizá DATASET_PATH para apuntar a genres_original."
    )

print(f"Dataset: {DATASET_PATH.resolve()}")

## 2. Índice y partición del dataset

Se separan canciones completas, no fragmentos de una misma canción. La partición es estratificada y reproducible: 80% entrenamiento, 10% validación y 10% test. La validación se usa para seleccionar el modelo; el test interno se reserva para la evaluación final del ganador.

In [ ]:
genre_dirs = sorted(
    folder for folder in DATASET_PATH.iterdir()
    if folder.is_dir() and not folder.name.startswith(".")
)
class_names = [folder.name for folder in genre_dirs]
class_to_idx = {name: idx for idx, name in enumerate(class_names)}

audio_paths, labels = [], []
for folder in genre_dirs:
    for audio_path in sorted(folder.glob("*.wav")):
        audio_paths.append(str(audio_path))
        labels.append(class_to_idx[folder.name])

if not audio_paths:
    raise FileNotFoundError(f"No hay archivos WAV en las carpetas de {DATASET_PATH}")

print(f"Géneros ({len(class_names)}): {class_names}")
print(f"Audios: {len(audio_paths)}")

train_paths, holdout_paths, train_labels, holdout_labels = train_test_split(
    audio_paths,
    labels,
    test_size=0.20,
    random_state=SEED,
    stratify=labels,
)
val_paths, test_paths, val_labels, test_labels = train_test_split(
    holdout_paths,
    holdout_labels,
    test_size=0.50,
    random_state=SEED,
    stratify=holdout_labels,
)

split_summary = pd.DataFrame({
    "Conjunto": ["Entrenamiento", "Validación", "Test"],
    "Cantidad": [len(train_paths), len(val_paths), len(test_paths)],
})
display(split_summary)
print("La semilla y el orden de clases quedan fijos para que la comparación sea reproducible.")

## 3. Extracción y normalización de MFCC

Se usan 40 MFCC, 22 050 Hz, FFT de 2048 y hop length de 512. La matriz de Librosa tiene forma `(40, T)`; se transpone a `(T, 40)`. Se estandariza cada coeficiente dentro de cada canción usando solo sus frames reales y después se ajusta a 1300 frames.

La extracción usa hasta cinco threads. El caché guarda las secuencias ya transformadas para no volver a calcularlas al reiniciar Colab. La normalización es por canción y no estima estadísticas sobre validación o test.

In [ ]:
SAMPLE_RATE = 22050
DURATION_SECONDS = 30
N_MFCC = 40
N_FFT = 2048
HOP_LENGTH = 512
MAX_FRAMES = 1300
CACHE_VERSION = 1
CACHE_DIR = Path.cwd() / ".mfcc_sequence_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

def _cache_path_for_audio(audio_path):
    signature = (
        f"{Path(audio_path).resolve()}|{SAMPLE_RATE}|{DURATION_SECONDS}|"
        f"{N_MFCC}|{N_FFT}|{HOP_LENGTH}|{MAX_FRAMES}|{CACHE_VERSION}"
    )
    key = hashlib.sha1(signature.encode("utf-8")).hexdigest()
    return CACHE_DIR / f"{key}.npz"

def extract_mfcc_sequence(audio_path):
    """Devuelve una secuencia float32 (MAX_FRAMES, N_MFCC) y su longitud real."""
    cache_path = _cache_path_for_audio(audio_path)
    if cache_path.is_file():
        try:
            with np.load(cache_path) as cached:
                if int(cached["version"].item()) == CACHE_VERSION:
                    sequence = cached["sequence"].astype(np.float32)
                    length = int(cached["length"].item())
                    if sequence.shape == (MAX_FRAMES, N_MFCC):
                        return sequence, length
        except (OSError, KeyError, ValueError):
            pass  # Caché incompleto: volver a calcular el audio.

    y, sr = librosa.load(
        audio_path,
        sr=SAMPLE_RATE,
        mono=True,
        duration=DURATION_SECONDS,
    )
    mfcc = librosa.feature.mfcc(
        y=y,
        sr=sr,
        n_mfcc=N_MFCC,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
    )

    # Librosa entrega (coeficientes, frames); la RNN recibe (frames, coeficientes).
    sequence = mfcc.T.astype(np.float32)
    sequence = sequence[:MAX_FRAMES]
    length = len(sequence)
    if length == 0:
        raise ValueError(f"No se pudieron extraer frames MFCC de {audio_path}")

    # Z-score por coeficiente, usando solo frames de audio y antes del padding.
    mean = sequence.mean(axis=0, keepdims=True)
    std = sequence.std(axis=0, keepdims=True)
    sequence = (sequence - mean) / np.maximum(std, 1e-6)

    if length < MAX_FRAMES:
        padding = np.zeros((MAX_FRAMES - length, N_MFCC), dtype=np.float32)
        sequence = np.concatenate([sequence, padding], axis=0)

    np.savez_compressed(
        cache_path,
        version=CACHE_VERSION,
        sequence=sequence.astype(np.float32),
        length=length,
    )
    return sequence.astype(np.float32), length

def extract_labeled_audio(item):
    audio_path, label = item
    try:
        sequence, length = extract_mfcc_sequence(audio_path)
        return audio_path, label, sequence, length
    except Exception as exc:
        print(f"Error procesando {audio_path}: {exc}")
        return None

# executor.map conserva el orden de entrada, por eso cada secuencia sigue asociada a su etiqueta.
with ThreadPoolExecutor(max_workers=5) as executor:
    extracted = list(tqdm(
        executor.map(extract_labeled_audio, zip(audio_paths, labels)),
        total=len(audio_paths),
        desc="Extrayendo MFCC",
        unit="audio",
    ))

valid_rows = [row for row in extracted if row is not None]
if not valid_rows:
    raise RuntimeError("No se pudo procesar ningún audio.")

audio_paths = [row[0] for row in valid_rows]
labels = [row[1] for row in valid_rows]
sequences = [row[2] for row in valid_rows]
sequence_lengths = [row[3] for row in valid_rows]

# Reindexar los splits por ruta tras descartar únicamente los archivos con error.
row_by_path = {path: idx for idx, path in enumerate(audio_paths)}
def indices_for(paths):
    return [row_by_path[path] for path in paths if path in row_by_path]

train_idx = indices_for(train_paths)
val_idx = indices_for(val_paths)
test_idx = indices_for(test_paths)

print(f"Secuencias: {len(sequences)} | forma por muestra: {sequences[0].shape}")
print(f"Longitud real promedio: {np.mean(sequence_lengths):.1f} frames")

In [ ]:
class MFCCSequenceDataset(Dataset):
    def __init__(self, indices):
        self.indices = list(indices)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, item):
        idx = self.indices[item]
        features = torch.from_numpy(sequences[idx])
        length = torch.tensor(sequence_lengths[idx], dtype=torch.long)
        label = torch.tensor(labels[idx], dtype=torch.long)
        return features, length, label

BATCH_SIZE = 16
train_dataset = MFCCSequenceDataset(train_idx)
val_dataset = MFCCSequenceDataset(val_idx)
test_dataset = MFCCSequenceDataset(test_idx)

def make_train_loader():
    generator = torch.Generator().manual_seed(SEED)
    return DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        generator=generator,
    )

val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Lotes: train={len(train_dataset)}, val={len(val_dataset)}, test={len(test_dataset)}")
print("Forma esperada por lote: (BatchSize, 1300, 40)")

## 4. Modelos recurrentes desde cero

Ambos modelos reciben `(Batch, T, 40)`, usan dos capas recurrentes bidireccionales, estado oculto de 128 y atención temporal. La RNN clásica usa `tanh`; la LSTM incorpora sus compuertas internas. Se empaquetan las secuencias según su longitud real para que el padding no altere la recurrencia ni la atención.

In [ ]:
class RecurrentGenreClassifier(nn.Module):
    def __init__(self, rnn_type, input_size=40, hidden_size=128, num_layers=2, dropout=0.3, num_classes=10):
        super().__init__()
        self.rnn_type = rnn_type.upper()
        if self.rnn_type == "RNN":
            self.rnn = nn.RNN(
                input_size=input_size,
                hidden_size=hidden_size,
                num_layers=num_layers,
                nonlinearity="tanh",
                batch_first=True,
                bidirectional=True,
                dropout=dropout,
            )
        elif self.rnn_type == "LSTM":
            self.rnn = nn.LSTM(
                input_size=input_size,
                hidden_size=hidden_size,
                num_layers=num_layers,
                batch_first=True,
                bidirectional=True,
                dropout=dropout,
            )
        else:
            raise ValueError("rnn_type debe ser 'RNN' o 'LSTM'.")

        recurrent_dim = hidden_size * 2
        self.attention = nn.Linear(recurrent_dim, 1)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(recurrent_dim, num_classes)

    def forward(self, features, lengths):
        packed = pack_padded_sequence(
            features,
            lengths.detach().cpu(),
            batch_first=True,
            enforce_sorted=False,
        )
        packed_out, _ = self.rnn(packed)
        recurrent_out, _ = pad_packed_sequence(
            packed_out,
            batch_first=True,
            total_length=features.size(1),
        )

        scores = self.attention(recurrent_out).squeeze(-1)
        time_steps = torch.arange(features.size(1), device=features.device).unsqueeze(0)
        valid_mask = time_steps < lengths.to(features.device).unsqueeze(1)
        scores = scores.masked_fill(~valid_mask, torch.finfo(scores.dtype).min)
        weights = torch.softmax(scores, dim=1)
        context = torch.sum(recurrent_out * weights.unsqueeze(-1), dim=1)
        logits = self.classifier(self.dropout(context))
        return logits, weights

def count_parameters(model):
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)

## 5. Entrenamiento y comparación

Se mantiene la misma partición y configuración para ambos modelos: AdamW, learning rate 0.001, weight decay 1e-4, cross entropy con label smoothing, scheduler ReduceLROnPlateau, clipping de gradiente y parada temprana. El checkpoint se elige con accuracy de validación; en empate se prioriza F1 macro. No se usan modelos preentrenados.

In [ ]:
EPOCHS = 40
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
EARLY_STOPPING_PATIENCE = 10
LABEL_SMOOTHING = 0.05

def run_epoch(model, loader, criterion, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss = 0.0
    y_true, y_pred = [], []

    for features, lengths, targets in loader:
        features = features.to(DEVICE)
        targets = targets.to(DEVICE)
        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):
            logits, _ = model(features, lengths)
            loss = criterion(logits, targets)
            if training:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
                optimizer.step()

        total_loss += loss.item() * targets.size(0)
        y_true.extend(targets.detach().cpu().numpy())
        y_pred.extend(logits.argmax(dim=1).detach().cpu().numpy())

    return {
        "loss": total_loss / len(loader.dataset),
        "accuracy": accuracy_score(y_true, y_pred),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }

def train_model(rnn_type):
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    model = RecurrentGenreClassifier(
        rnn_type=rnn_type,
        num_classes=len(class_names),
    ).to(DEVICE)
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=3,
    )

    history = {key: [] for key in (
        "train_loss", "val_loss", "train_accuracy", "val_accuracy",
        "train_f1", "val_f1",
    )}
    best_key = (-1.0, -1.0)
    best_epoch = 0
    stale_epochs = 0
    checkpoint = Path(f"best_{rnn_type.lower()}_mfcc.pt")
    train_loader = make_train_loader()

    for epoch in range(EPOCHS):
        train_metrics = run_epoch(model, train_loader, criterion, optimizer)
        val_metrics = run_epoch(model, val_loader, criterion)
        scheduler.step(val_metrics["loss"])

        history["train_loss"].append(train_metrics["loss"])
        history["val_loss"].append(val_metrics["loss"])
        history["train_accuracy"].append(train_metrics["accuracy"])
        history["val_accuracy"].append(val_metrics["accuracy"])
        history["train_f1"].append(train_metrics["f1_macro"])
        history["val_f1"].append(val_metrics["f1_macro"])

        current_key = (val_metrics["accuracy"], val_metrics["f1_macro"])
        if current_key > best_key:
            best_key = current_key
            best_epoch = epoch + 1
            stale_epochs = 0
            torch.save(model.state_dict(), checkpoint)
        else:
            stale_epochs += 1

        print(
            f"[{rnn_type}] época {epoch + 1:02d}/{EPOCHS} | "
            f"train loss {train_metrics['loss']:.4f}, acc {train_metrics['accuracy']:.3f}, "
            f"F1 {train_metrics['f1_macro']:.3f} | "
            f"val loss {val_metrics['loss']:.4f}, acc {val_metrics['accuracy']:.3f}, "
            f"F1 {val_metrics['f1_macro']:.3f} | "
            f"LR {optimizer.param_groups[0]['lr']:.6g}"
        )
        if stale_epochs >= EARLY_STOPPING_PATIENCE:
            print(f"Early stopping: sin mejora de val accuracy durante {stale_epochs} épocas.")
            break

    model.load_state_dict(torch.load(checkpoint, map_location=DEVICE))
    return model, history, best_epoch, checkpoint

models, histories, best_epochs, checkpoints = {}, {}, {}, {}
for model_name in ("RNN", "LSTM"):
    print(f"\nEntrenando {model_name} desde cero")
    models[model_name], histories[model_name], best_epochs[model_name], checkpoints[model_name] = train_model(model_name)

In [ ]:
# Curvas por modelo: accuracy, F1 macro y loss de entrenamiento/validación.
fig, axes = plt.subplots(2, 3, figsize=(17, 9))
metrics_to_plot = [
    ("accuracy", "Accuracy"),
    ("f1", "F1 macro"),
    ("loss", "Loss"),
]
for row, model_name in enumerate(("RNN", "LSTM")):
    history = histories[model_name]
    for col, (metric, title) in enumerate(metrics_to_plot):
        axes[row, col].plot(history[f"train_{metric}"], label="Train")
        axes[row, col].plot(history[f"val_{metric}"], label="Validación")
        axes[row, col].set_title(f"{model_name}: {title}")
        axes[row, col].set_xlabel("Época")
        axes[row, col].grid(True, alpha=0.3)
        axes[row, col].legend()
plt.tight_layout()
plt.show()

In [ ]:
def collect_predictions(model, loader):
    model.eval()
    all_targets, all_predictions, all_probabilities = [], [], []
    with torch.no_grad():
        for features, lengths, targets in loader:
            logits, _ = model(features.to(DEVICE), lengths)
            probabilities = torch.softmax(logits, dim=1)
            all_targets.extend(targets.numpy())
            all_predictions.extend(logits.argmax(dim=1).cpu().numpy())
            all_probabilities.extend(probabilities.cpu().numpy())
    return (
        np.asarray(all_targets),
        np.asarray(all_predictions),
        np.asarray(all_probabilities),
    )

validation_results = {}
for model_name, model in models.items():
    y_val, pred_val, prob_val = collect_predictions(model, val_loader)
    val_accuracy = accuracy_score(y_val, pred_val)
    val_f1 = f1_score(y_val, pred_val, average="macro", zero_division=0)
    validation_results[model_name] = {
        "Accuracy validación": val_accuracy,
        "F1 macro validación": val_f1,
        "Época del checkpoint": best_epochs[model_name],
        "Parámetros": count_parameters(model),
    }

    print(f"\n===== {model_name} — validación =====")
    print(f"Accuracy: {val_accuracy:.4f}")
    print(f"F1 macro: {val_f1:.4f}")
    print(classification_report(
        y_val, pred_val, labels=np.arange(len(class_names)),
        target_names=class_names, digits=3, zero_division=0,
    ))

    cm = confusion_matrix(y_val, pred_val, labels=np.arange(len(class_names)))
    cm_norm = confusion_matrix(
        y_val, pred_val, labels=np.arange(len(class_names)), normalize="true"
    )
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names, ax=axes[0])
    axes[0].set_title(f"{model_name} — confusión absoluta")
    sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1,
                xticklabels=class_names, yticklabels=class_names, ax=axes[1])
    axes[1].set_title(f"{model_name} — confusión normalizada por fila")
    for ax in axes:
        ax.set_xlabel("Predicción")
        ax.set_ylabel("Género real")
        ax.tick_params(axis="x", rotation=45)
    plt.tight_layout()
    plt.show()

comparison = pd.DataFrame(validation_results).T.sort_values(
    ["Accuracy validación", "F1 macro validación"], ascending=False
)
display(comparison.style.format({
    "Accuracy validación": "{:.2%}",
    "F1 macro validación": "{:.2%}",
    "Época del checkpoint": "{:.0f}",
    "Parámetros": "{:,.0f}",
}))

# El ganador se define con validación; el test no interviene en esta decisión.
winner = comparison.index[0]
winner_model = models[winner]
print(f"Modelo ganador según validación: {winner}")

## 6. Evaluación interna final en test

El conjunto test se consulta después de elegir el ganador en validación. Se reportan accuracy, F1 macro, classification report y matrices de confusión. No se usa este resultado para volver a elegir entre RNN y LSTM.

In [ ]:
y_test, pred_test, prob_test = collect_predictions(winner_model, test_loader)
test_accuracy = accuracy_score(y_test, pred_test)
test_f1 = f1_score(y_test, pred_test, average="macro", zero_division=0)

print(f"Ganador: {winner}")
print(f"Accuracy test: {test_accuracy:.4f}")
print(f"F1 macro test: {test_f1:.4f}")
print(classification_report(
    y_test, pred_test, labels=np.arange(len(class_names)),
    target_names=class_names, digits=3, zero_division=0,
))

cm_test = confusion_matrix(y_test, pred_test, labels=np.arange(len(class_names)))
cm_test_norm = confusion_matrix(
    y_test, pred_test, labels=np.arange(len(class_names)), normalize="true"
)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.heatmap(cm_test, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names, ax=axes[0])
axes[0].set_title("Test — confusión absoluta")
sns.heatmap(cm_test_norm, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1,
            xticklabels=class_names, yticklabels=class_names, ax=axes[1])
axes[1].set_title("Test — confusión normalizada por fila")
for ax in axes:
    ax.set_xlabel("Predicción")
    ax.set_ylabel("Género real")
    ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

## 7. Generalización a canciones nuevas

Las canciones de esta sección deben ser externas a GTZAN. Colocá al menos 10 WAVs en `canciones_nuevas/`, con al menos una canción representativa de cada género. Completá el CSV generado por la celda siguiente: `file` puede ser el nombre del archivo o una ruta relativa a esa carpeta, y `expected_genre` debe coincidir con una de las carpetas de GTZAN.

El código valida cantidad, géneros, existencia de archivos y que no se use una ruta dentro de GTZAN. No descarga canciones ni inventa etiquetas: hay que aportar audios externos y sus géneros esperados antes de ejecutar la evaluación.

In [ ]:
EXTERNAL_DIR = Path("canciones_nuevas")
EXTERNAL_DIR.mkdir(parents=True, exist_ok=True)
EXTERNAL_MANIFEST = EXTERNAL_DIR / "external_manifest.csv"

if not EXTERNAL_MANIFEST.exists():
    template = pd.DataFrame({
        "file": [""] * len(class_names),
        "expected_genre": class_names,
    })
    template.to_csv(EXTERNAL_MANIFEST, index=False)
    print(f"Creé la plantilla: {EXTERNAL_MANIFEST}")
    print("Agregá los WAV externos y completá una fila por género; luego volvé a ejecutar esta celda.")
else:
    external_manifest = pd.read_csv(EXTERNAL_MANIFEST)
    print(f"Manifiesto encontrado: {len(external_manifest)} filas")
    display(external_manifest.head())

In [ ]:
external_predictions = None
manifest_ready = False
if EXTERNAL_MANIFEST.exists():
    manifest_preview = pd.read_csv(EXTERNAL_MANIFEST)
    manifest_ready = (
        "file" in manifest_preview.columns
        and manifest_preview["file"].fillna("").astype(str).str.strip().ne("").any()
    )


if manifest_ready:
    external_manifest = pd.read_csv(EXTERNAL_MANIFEST).dropna(subset=["file", "expected_genre"])
    external_manifest["file"] = external_manifest["file"].astype(str).str.strip()
    external_manifest["expected_genre"] = external_manifest["expected_genre"].astype(str).str.strip()
    external_manifest = external_manifest[external_manifest["file"] != ""].copy()

    if len(external_manifest) < 10:
        raise ValueError(f"Se requieren al menos 10 canciones externas; hay {len(external_manifest)}.")
    genre_lookup = {genre.lower(): genre for genre in class_names}
    external_manifest["expected_genre"] = external_manifest["expected_genre"].map(
        lambda value: genre_lookup.get(value.lower(), value)
    )
    unknown_genres = sorted(set(external_manifest["expected_genre"]) - set(class_names))
    if unknown_genres:
        raise ValueError(f"Géneros del manifiesto no reconocidos: {unknown_genres}")
    missing_genres = sorted(set(class_names) - set(external_manifest["expected_genre"]))
    if missing_genres:
        raise ValueError(f"Falta al menos una canción esperada para estos géneros: {missing_genres}")

    external_paths = []
    for filename in external_manifest["file"]:
        candidate = Path(filename)
        path = candidate if candidate.is_absolute() else EXTERNAL_DIR / candidate
        path = path.resolve()
        if not path.is_file():
            raise FileNotFoundError(f"No existe el audio externo: {path}")
        try:
            path.relative_to(DATASET_PATH.resolve())
        except ValueError:
            pass
        else:
            raise ValueError(f"El audio debe estar fuera de GTZAN: {path}")
        external_paths.append(path)

    if len(set(external_paths)) < 10:
        raise ValueError("El manifiesto debe apuntar a por lo menos 10 archivos distintos.")

    external_rows = []
    winner_model.eval()
    for row, path in tqdm(
        zip(external_manifest.itertuples(index=False), external_paths),
        total=len(external_paths),
        desc="Evaluando canciones nuevas",
    ):
        sequence, length = extract_mfcc_sequence(path)
        features = torch.from_numpy(sequence).unsqueeze(0).to(DEVICE)
        lengths = torch.tensor([length], dtype=torch.long)
        with torch.no_grad():
            logits, _ = winner_model(features, lengths)
            probabilities = torch.softmax(logits, dim=1)[0].cpu().numpy()

        predicted_idx = int(np.argmax(probabilities))
        result = {
            "Canción": path.stem,
            "Género esperado": row.expected_genre,
            "Predicción": class_names[predicted_idx],
            "Correcta": class_names[predicted_idx] == row.expected_genre,
        }
        result.update({
            f"Score {genre} (%)": round(float(probabilities[idx] * 100), 2)
            for idx, genre in enumerate(class_names)
        })
        external_rows.append(result)

    external_predictions = pd.DataFrame(external_rows)
    display(external_predictions)
    print(
        "Accuracy externa descriptiva: "
        f"{external_predictions['Correcta'].mean():.2%} "
        f"({external_predictions['Correcta'].sum()}/{len(external_predictions)})"
    )

    # Un reproductor por canción para revisar el audio que produjo la predicción.
    from IPython.display import Audio, Markdown, display
    for idx, (row, path) in enumerate(zip(external_manifest.itertuples(index=False), external_paths)):
        display(Markdown(
            f"### {path.stem} — esperado: **{row.expected_genre}** | "
            f"predicho: **{external_predictions.iloc[idx, 2]}**"
        ))
        display(Audio(filename=str(path)))
else:
    print("Completá external_manifest.csv y volvé a ejecutar esta celda.")

## 8. Análisis de errores y conclusión

La predicción externa no determina por sí sola la causa de un error. Para cada canción mal clasificada, escuchá el audio y revisá los scores. Considerá similitud entre géneros, mezcla de estilos, calidad/recorte del audio y cobertura limitada de GTZAN.

Completá el diagnóstico con observaciones basadas en cada canción, no en supuestos. En la conclusión, compará RNN y LSTM usando métricas de validación; informá el resultado final del ganador en test interno y describí con cautela el desempeño en canciones nuevas.

In [ ]:
if external_predictions is not None:
    errors_external = external_predictions.loc[~external_predictions["Correcta"]].copy()
    if errors_external.empty:
        print("No hubo errores en las canciones nuevas del manifiesto.")
    else:
        display(errors_external[[
            "Canción", "Género esperado", "Predicción",
            *[f"Score {genre} (%)" for genre in class_names],
        ]])
        diagnostico = errors_external[["Canción", "Género esperado", "Predicción"]].copy()
        diagnostico["Posible causa (a completar escuchando el audio)"] = ""
        display(diagnostico)

print(
    f"Ganador por validación: {winner}. "
    f"Accuracy/F1 macro en test interno: {test_accuracy:.2%}/{test_f1:.2%}."
)
if external_predictions is None:
    print("Pendiente: agregar canciones externas y completar el manifiesto para cerrar la evaluación de generalización.")
else:
    print(
        f"Resultado externo descriptivo: "
        f"{external_predictions['Correcta'].mean():.2%} de aciertos sobre "
        f"{len(external_predictions)} canciones."
    )